# Caustic-position convergence with solver tolerance

This notebook repeats the single-ray, 20-degree linear-density example while varying the dimensionless Diffrax tolerances. The state is automatically scaled before error control: propagation lengths use the resolved physical grid length, primary momenta use unity, neighbour positions use their initial offset, and neighbour momenta use their initial relative scale.

The plotted error is measured against the analytical turning point, $x_{\rm turn}=L_n\cos^2\alpha$. The absolute tolerance is kept at $10^{-2}$ times the relative tolerance. At tight tolerances the result may plateau because the caustic search also uses a finite diagnostic sampling grid. Each trace is also timed after synchronising JAX; the first measurement can include compilation overhead, so these are representative end-to-end timings rather than a formal benchmark.

In [ ]:
import time
from dataclasses import replace
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np

from pyGATH.io import load_simulation_config
from pyGATH.raytracing import RAY_STATE_LAYOUT, trace_rays

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent

simulation = load_simulation_config(
    repo_root / "configs" / "test_configs" / "linear_gradient_turning.toml"
)
grid = simulation.build_grid()
initial_rays = simulation.initialize_rays(grid)
base_options = simulation.raytracing.options()

density_length_m = 1.0e-3
incidence_angle_rad = np.deg2rad(20.0)
expected_caustic_x = density_length_m * np.cos(incidence_angle_rad) ** 2
expected_caustic_x

In [ ]:
relative_tolerances = np.array([1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6, 1e-6])
absolute_tolerances = 1e-2 * relative_tolerances
caustic_x = []
trace_times_s = []

for rtol, atol in zip(relative_tolerances, absolute_tolerances, strict=True):
    options = replace(
        base_options,
        rtol=float(rtol),
        atol=float(atol),
        diagnostic_samples=2048,
    )
    trace_start = time.perf_counter()
    result = trace_rays(initial_rays, grid, options=options)
    # JAX dispatch can be asynchronous on accelerators, so wait for the
    # traced fields before stopping the wall-clock timer.
    jax.block_until_ready(result.sheet_fields)
    trace_times_s.append(time.perf_counter() - trace_start)
    if not bool(result.has_caustic[0, 0, 0]):
        raise RuntimeError(f"No caustic found for rtol={rtol:g}, atol={atol:g}")
    caustic_x.append(
        float(result.sheet_fields[0, 0, 0, 0, -1, RAY_STATE_LAYOUT.position.start])
    )

caustic_x = np.asarray(caustic_x)
trace_times_s = np.asarray(trace_times_s)
absolute_error = np.abs(caustic_x - expected_caustic_x)
relative_error = absolute_error / expected_caustic_x

for rtol, atol, location, error, elapsed in zip(
    relative_tolerances,
    absolute_tolerances,
    caustic_x,
    absolute_error,
    trace_times_s,
    strict=True,
):
    print(
        f"rtol={rtol:8.1e}, atol={atol:8.1e}: "
        f"x_caustic={location:.9e} m, |error|={error:.3e} m, "
        f"trace time={elapsed:.3f} s"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_floor = np.finfo(float).eps

axes[0].loglog(relative_tolerances, np.maximum(absolute_error, plot_floor), marker="o")
axes[0].set_xlabel("Dimensionless relative tolerance")
axes[0].set_ylabel("Absolute caustic-position error [m]")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].invert_xaxis()

axes[1].loglog(relative_tolerances, np.maximum(relative_error, plot_floor), marker="o")
axes[1].set_xlabel("Dimensionless relative tolerance")
axes[1].set_ylabel("Relative caustic-position error")
axes[1].grid(True, which="both", alpha=0.3)
axes[1].invert_xaxis()

axes[2].semilogx(relative_tolerances, trace_times_s, marker="o")
axes[2].set_xlabel("Dimensionless relative tolerance")
axes[2].set_ylabel("trace_rays wall time [s]")
axes[2].grid(True, which="both", alpha=0.3)
axes[2].invert_xaxis()

fig.suptitle("Linear-gradient caustic convergence (atol = 0.01 rtol)")
fig.tight_layout()
plt.show()